VIDEO TO FRAMES

In [1]:
import cv2
import os

video_path = "sample_video.mp4"
output_folder = "frames"

os.makedirs(output_folder, exist_ok=True)

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)

metadata_list = []
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    if frame_count % 12 == 0 and frame_count != 0:
        filename = f"frame_{frame_count:06d}.jpg"
        filepath = f"{output_folder}/{filename}"
        
        cv2.imwrite(filepath, frame)
        
        timestamp = frame_count / fps
        
        metadata_list.append({
            "frame_index": frame_count,
            "timestamp": round(timestamp, 2),
            "image_path": filepath
        })
        
    frame_count += 1

cap.release()
print(f"Đã trích xuất {len(metadata_list)} frames vào thư mục {output_folder}")
print("Metadata mẫu của frame đầu tiên:", metadata_list[0] if metadata_list else "Không có data")

Đã trích xuất 150 frames vào thư mục frames
Metadata mẫu của frame đầu tiên: {'frame_index': 12, 'timestamp': 0.4, 'image_path': 'frames/frame_000012.jpg'}


FRAME TO VECTOR

In [ ]:
import torch
import faiss
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

print("Downloading CLIP...")
model_id = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_id)
processor = CLIPProcessor.from_pretrained(model_id)

image_embeddings = []

print("Bắt đầu mã hóa (encode) các frame thành vector...")
for meta in metadata_list:
    image = Image.open(meta["image_path"])
    
    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        features = model.get_image_features(**inputs)
        
        if not isinstance(features, torch.Tensor):
            features = features.pooler_output if hasattr(features, 'pooler_output') else features[0]
            
    embedding = features.cpu().numpy()
    
    faiss.normalize_L2(embedding)
    image_embeddings.append(embedding[0])

image_embeddings = np.array(image_embeddings).astype('float32')
print(f"Kích thước ma trận embedding: {image_embeddings.shape}")

dimension = image_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension) 

index.add(image_embeddings)
print(f"Đã thêm {index.ntotal} vector vào FAISS index.")

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 651.60it/s]


Bắt đầu mã hóa (encode) các frame thành vector...
Kích thước ma trận embedding: (150, 512)
Đã thêm 150 vector vào FAISS index.


QUERY -> VECTOR -> SEARCH -> RESULT

In [ ]:
query_text = "a corgi puppy in a gift box" 
print(f"Top 10 results for: {query_text}")

text_inputs = processor(text=[query_text], return_tensors="pt", padding=True)
with torch.no_grad():
    text_features = model.get_text_features(**text_inputs)
    
    if not isinstance(text_features, torch.Tensor):
        text_features = text_features.pooler_output if hasattr(text_features, 'pooler_output') else text_features[0]
        
text_embedding = text_features.cpu().numpy()
faiss.normalize_L2(text_embedding)

k = 10
scores, indices = index.search(text_embedding, k)

for i in range(k):
    match_idx = indices[0][i]
    
    meta = metadata_list[match_idx]
    
    rank = i + 1
    frame = meta['frame_index']
    time = meta['timestamp']
    score = scores[0][i]
    path = meta['image_path']
    
    print(f"{rank}. frame={frame} time={time:.2f}s score={score:.3f} path={path}")

Top 10 results for: a corgi puppy in a gift box
1. frame=1272 time=42.40s score=0.317 path=frames/frame_001272.jpg
2. frame=972 time=32.40s score=0.314 path=frames/frame_000972.jpg
3. frame=948 time=31.60s score=0.314 path=frames/frame_000948.jpg
4. frame=1284 time=42.80s score=0.309 path=frames/frame_001284.jpg
5. frame=984 time=32.80s score=0.308 path=frames/frame_000984.jpg
6. frame=936 time=31.20s score=0.306 path=frames/frame_000936.jpg
7. frame=960 time=32.00s score=0.306 path=frames/frame_000960.jpg
8. frame=324 time=10.80s score=0.305 path=frames/frame_000324.jpg
9. frame=1596 time=53.20s score=0.301 path=frames/frame_001596.jpg
10. frame=1584 time=52.80s score=0.298 path=frames/frame_001584.jpg
